<a href="https://colab.research.google.com/github/polreig/StartUp_DecoAI/blob/main/4_Motor_intereactivo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Instalación

In [ ]:
!pip install -q diffusers transformers accelerate opencv-python Pillow gradio

App Web

In [ ]:
import gradio as gr
import torch
import cv2
import numpy as np
from PIL import Image
from diffusers import StableDiffusionControlNetInpaintPipeline, ControlNetModel, DDIMScheduler

print("🚀 Cargando el Motor Gráfico... (Esto tomará un minuto)")

# 1. CARGA DE MODELOS (Ya no necesitamos el de segmentación automática)
controlnet = ControlNetModel.from_pretrained("lllyasviel/control_v11p_sd15_mlsd", torch_dtype=torch.float16)
pipe = StableDiffusionControlNetInpaintPipeline.from_pretrained(
    "runwayml/stable-diffusion-inpainting",
    controlnet=controlnet,
    torch_dtype=torch.float16
).to("cuda")
pipe.scheduler = DDIMScheduler.from_config(pipe.scheduler.config)

# 2. FUNCIONES INTERNAS
def redimensionar(img, max_size=512):
    ancho, alto = img.size
    ratio = alto / ancho
    nuevo_ancho, nuevo_alto = (max_size, int(max_size * ratio)) if ancho > alto else (int(max_size / ratio), max_size)
    nuevo_ancho, nuevo_alto = (nuevo_ancho // 8) * 8, (nuevo_alto // 8) * 8
    return img.resize((nuevo_ancho, nuevo_alto), Image.Resampling.LANCZOS)

def extraer_mlsd(img):
    img_gray = cv2.cvtColor(np.array(img), cv2.COLOR_RGB2GRAY)
    lsd = cv2.createLineSegmentDetector(0)
    lines, _, _, _ = lsd.detect(img_gray)
    drawn_img = np.zeros_like(img_gray)
    if lines is not None: lsd.drawSegments(drawn_img, lines)
    return Image.fromarray(drawn_img).convert("RGB")

# 3. EL CEREBRO DE LA APP (Actualizado para Gradio 4/5)
def procesar_imagen(dict_imagen, peticion_usuario, fuerza):
    if dict_imagen is None or dict_imagen["background"] is None:
        return None
        
    print("🎨 Recibiendo tu dibujo y procesando...")

    # El nuevo Gradio separa la foto original ('background') de los trazos del pincel ('layers')
    img_original = dict_imagen["background"].convert("RGB")
    
    # Extraemos la máscara usando la transparencia (canal Alpha) de lo que has dibujado
    if len(dict_imagen["layers"]) > 0:
        img_mascara = dict_imagen["layers"][0].split()[-1].convert("L")
    else:
        # Si le das al botón sin haber pintado nada, crea una máscara vacía para no fallar
        img_mascara = Image.new("L", img_original.size, 0)

    # Ajustamos tamaños
    img_original = redimensionar(img_original)
    img_mascara = img_mascara.resize(img_original.size, Image.Resampling.LANCZOS)

    # Extraemos la estructura arquitectónica
    img_mlsd = extraer_mlsd(img_original)

    # Renderizamos quirúrgicamente
    neg_prompt = "cartoon, illustration, low quality, warped lines, messy, unrealistic lighting, deformed, outdoors"
    
    resultado = pipe(
        prompt=peticion_usuario + ", photorealistic, architectural digest, 8k resolution, highly detailed",
        negative_prompt=neg_prompt,
        image=img_original,
        mask_image=img_mascara,
        control_image=img_mlsd,
        num_inference_steps=30,
        controlnet_conditioning_scale=1.0, # Mantiene la estructura estricta
        strength=fuerza, # Cuánta libertad tiene para cambiar la zona pintada
        guidance_scale=8.5
    ).images[0]

    return resultado

# 4. LA INTERFAZ WEB (Actualizada para Gradio 4/5)
print("🌐 Creando tu página web...")
with gr.Blocks(theme=gr.themes.Soft()) as app:
    gr.Markdown("# ✨ DecoAI: Modo Quirúrgico Profesional")
    gr.Markdown("1. **Sube tu foto**. 2. **Pinta** con el pincel por encima del mueble que quieres cambiar. 3. **Escribe** lo que quieres poner ahí.")
    
    with gr.Row():
        with gr.Column():
            # ¡EL NUEVO COMPONENTE MAGICO!
            editor_imagen = gr.ImageEditor(type="pil", label="Sube tu foto y pinta el mueble")
            prompt_input = gr.Textbox(label="¿Qué nuevo mueble quieres? (Ej: Una cama moderna con cabecero gris oscuro)", placeholder="A modern bed with dark grey upholstered headboard")
            slider_fuerza = gr.Slider(minimum=0.5, maximum=1.0, value=0.85, step=0.05, label="Fuerza del cambio (0.85 recomendado)")
            boton_generar = gr.Button("🎨 Re-Diseñar Mueble", variant="primary")
            
        with gr.Column():
            imagen_salida = gr.Image(label="Tu Nuevo Diseño")
            
    boton_generar.click(fn=procesar_imagen, inputs=[editor_imagen, prompt_input, slider_fuerza], outputs=imagen_salida)

# Esto lanzará un enlace público
app.launch(share=True, debug=True)